# 01 — Dataset audit and BI-RADS label check

**Objective.** Establish what the BUSI copy in `dataset/BUSI_Jpeg/` actually contains before any
radiomics or model work: class labels, masks, image geometry, the existing train/val/test split,
exact and near-duplicate images, cross-split leakage, and — critically — whether genuine BI-RADS
reference labels exist.

**Prerequisites.** Stage 1 preprocessing has been run (`python train.py preprocess`), so
`dataset/splits/{train,val,test}.json` and `*_lesions.json` exist.

**Outputs.** `results/audit/dataset_audit.json` (machine-readable) — consumed by notebooks 02 and 04.

**Run order.** 01 → 02 → 03 → 04 → 05 → 06. This notebook trains nothing and is safe to re-run.

## Setup

In [1]:
import json
import os
import sys
from pathlib import Path

# Resolve the repo root whether the kernel starts in notebooks/ or at the root:
# config.yaml paths (dataset/, checkpoints/, results/) are relative to it.
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "config.yaml").exists() and (p / "bus_research").is_dir())
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils.config import load_config
from utils.seed import get_device, set_seed

config = load_config(str(REPO_ROOT / "config.yaml"))
set_seed(config["seed"], config["deterministic"])
RESEARCH = config["research"]
print(f"Repo root: {REPO_ROOT}")

Repo root: /Users/mulatyazewchekol/Documents/My_Projects/Healthcare_Projects/Breast_Cancer_Detection


In [2]:
import bus_research

missing = bus_research.missing_dependencies()
print("Missing research dependencies:", missing or "none")
# If anything is missing, install the pinned optional set (see the comments in that file):
# %pip install -r requirements-research.txt

Missing research dependencies: none


## 1. Images, masks and class labels

In [3]:
from collections import Counter

import cv2
import numpy as np
import pandas as pd

from dataset.manifest import _stem_to_image_id, build_manifest, file_hash

P = config["paths"]
records = build_manifest(P["raw_benign"], P["raw_benign_mask"], P["raw_malignant"], P["raw_malignant_mask"])
label_counts = Counter(r.label for r in records)
lesion_counts = Counter(len(r.mask_paths) for r in records)
no_mask = [r.image_id for r in records if not r.mask_paths]

orphan_masks = []
for mask_dir in (P["raw_benign_mask"], P["raw_malignant_mask"]):
    ids = {r.image_id for r in records}
    orphan_masks += [m.name for m in sorted(Path(mask_dir).glob("*.png")) if _stem_to_image_id(m.stem) not in ids]

print("images per class:", dict(label_counts))
print("mask files per image:", dict(sorted(lesion_counts.items())))
print("images without a mask:", no_mask or "none")
print("mask files without an image:", orphan_masks or "none")
print("classes present:", sorted(label_counts), "(no 'normal' class in this copy)")

images per class: {'benign': 437, 'malignant': 210}
mask files per image: {1: 630, 2: 16, 3: 1}
images without a mask: none
mask files without an image: none
classes present: ['benign', 'malignant'] (no 'normal' class in this copy)


In [4]:
geom = []
for r in records:
    img = cv2.imread(str(r.image_path), cv2.IMREAD_UNCHANGED)
    gray_in_rgb = img.ndim == 3 and bool((img[..., 0] == img[..., 1]).all() and (img[..., 1] == img[..., 2]).all())
    geom.append({"label": r.label, "h": img.shape[0], "w": img.shape[1], "channels": 1 if img.ndim == 2 else img.shape[2],
                 "dtype": str(img.dtype), "gray_in_rgb": gray_in_rgb})
geom = pd.DataFrame(geom)
print(geom.groupby("label")[["h", "w"]].describe().round(1).T)
print("\ndtypes:", geom.dtype.value_counts().to_dict(), "| channels:", geom.channels.value_counts().to_dict())
print("grayscale stored as identical RGB channels:", f"{geom.gray_in_rgb.mean():.1%}")

label    benign  malignant
h count   437.0      210.0
  mean    495.5      494.0
  std      78.1       60.4
  min     324.0      332.0
  25%     464.0      463.0
  50%     471.0      474.0
  75%     574.0      554.0
  max     719.0      677.0
w count   437.0      210.0
  mean    613.4      598.0
  std     117.3      125.2
  min     315.0      190.0
  25%     554.0      552.0
  50%     563.0      564.0
  75%     706.0      649.8
  max    1028.0     1048.0

dtypes: {'uint8': 647} | channels: {3: 647}
grayscale stored as identical RGB channels: 97.7%


Image sizes vary per image, so all radiomics work in notebook 02 runs on each image at its
**native resolution**. BUSI ships no pixel-spacing metadata, so physical units are unavailable.

## 2. Existing split: coverage and exact-duplicate leakage

In [5]:
from dataset.splits import load_split

splits = {s: load_split(P["splits_dir"], s) for s in ("train", "val", "test")}
split_of = {}
for s, rows in splits.items():
    for row in rows:
        assert row["image_id"] not in split_of, f"{row['image_id']} appears in two splits"
        split_of[row["image_id"]] = s
manifest_ids = {r.image_id for r in records}
print("split sizes:", {s: len(rows) for s, rows in splits.items()})
print("per class:", {s: dict(Counter(r["label"] for r in rows)) for s, rows in splits.items()})
print("every manifest image is in exactly one split:", set(split_of) == manifest_ids)

hash_of = {r.image_id: file_hash(r.image_path) for r in records}
groups = {}
for image_id, h in hash_of.items():
    groups.setdefault(h, []).append(image_id)
exact_dups = [sorted(g) for g in groups.values() if len(g) > 1]
cross_split_exact = [g for g in exact_dups if len({split_of[i] for i in g}) > 1]
print("\nbyte-identical groups:", exact_dups)
print("byte-identical groups spanning splits:", cross_split_exact or "none")

split sizes: {'train': 452, 'val': 98, 'test': 97}
per class: {'train': {'benign': 306, 'malignant': 146}, 'val': {'benign': 65, 'malignant': 33}, 'test': {'benign': 66, 'malignant': 31}}
every manifest image is in exactly one split: True

byte-identical groups: [['benign (433)', 'malignant (145)']]
byte-identical groups spanning splits: none


## 3. Near-duplicate images (perceptual hash)

BUSI has no patient identifiers, so different frames from the same patient/scan may sit in different
splits. A difference hash (dHash, 64 bits) plus a pixel-correlation check flags visually near-identical
images. This can only *detect* likely duplicates; it cannot recover patient identity.

In [6]:
def dhash_bits(path, size=8):
    g = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    r = cv2.resize(g, (size + 1, size), interpolation=cv2.INTER_AREA).astype(np.int16)
    return (r[:, 1:] > r[:, :-1]).ravel()

def thumb(path, size=64):
    g = cv2.resize(cv2.imread(str(path), cv2.IMREAD_GRAYSCALE), (size, size), interpolation=cv2.INTER_AREA)
    g = g.astype(np.float32).ravel()
    return (g - g.mean()) / (g.std() + 1e-6)

ids = [r.image_id for r in records]
bits = np.stack([dhash_bits(r.image_path) for r in records])
thumbs = np.stack([thumb(r.image_path) for r in records])
hamming = (bits[:, None, :] != bits[None, :, :]).sum(-1)
corr = thumbs @ thumbs.T / thumbs.shape[1]
iu = np.triu_indices(len(ids), k=1)
cand = [(ids[i], ids[j], int(hamming[i, j]), float(corr[i, j])) for i, j in zip(*iu)
        if hamming[i, j] <= 6 and corr[i, j] >= 0.95 and hash_of[ids[i]] != hash_of[ids[j]]]
near = pd.DataFrame(cand, columns=["image_a", "image_b", "dhash_hamming", "pixel_corr"])
if len(near):
    near["split_a"] = near.image_a.map(split_of)
    near["split_b"] = near.image_b.map(split_of)
    near["cross_split"] = near.split_a != near.split_b
    near["cross_label"] = near.image_a.str.split(" ").str[0] != near.image_b.str.split(" ").str[0]
print(f"near-duplicate pairs (hamming<=6 and corr>=0.95, not byte-identical): {len(near)}")
print(near.sort_values("pixel_corr", ascending=False).head(15).to_string(index=False) if len(near) else "")
print("\ncross-split near-duplicate pairs:", int(near.cross_split.sum()) if len(near) else 0)
print("cross-LABEL near-duplicate pairs (same-looking frame, different class folder):",
      int(near.cross_label.sum()) if len(near) else 0)

near-duplicate pairs (hamming<=6 and corr>=0.95, not byte-identical): 122
        image_a        image_b  dhash_hamming  pixel_corr split_a split_b  cross_split  cross_label
   benign (233)   benign (299)              1    0.999520    test   train         True        False
malignant (116) malignant (18)              5    0.998979   train   train        False        False
  malignant (4)  malignant (5)              3    0.998778     val   train         True        False
   benign (284)   benign (285)              3    0.998119     val   train         True        False
   benign (152)    benign (18)              0    0.997951   train   train        False        False
   benign (203)   benign (289)              0    0.997694   train   train        False        False
   benign (273)   benign (289)              0    0.996886   train   train        False        False
     benign (3)   benign (320)              1    0.996481   train   train        False        False
   benign (228)   benign (

## 4. Lesion-level manifests and identifier leakage

In [7]:
lesions = {s: json.loads((Path(P["splits_dir"]) / f"{s}_lesions.json").read_text()) for s in splits}
print("lesions per split:", {s: len(v) for s, v in lesions.items()})
print("lesions per class:", {s: dict(Counter(r["label"] for r in v)) for s, v in lesions.items()})
lesion_split_ok = all(split_of[r["image_id"]] == s for s, v in lesions.items() for r in v)
print("every lesion sits in its image's split:", lesion_split_ok)
label_in_id = all(r.image_id.startswith(r.label) for r in records)
print("raw image ids contain the class label:", label_in_id,
      "-> feature records must use salted pseudonymous ids (bus_research.feature_store.pseudonymize)")

lesions per split: {'train': 461, 'val': 104, 'test': 100}
lesions per class: {'train': {'benign': 315, 'malignant': 146}, 'val': {'benign': 70, 'malignant': 34}, 'test': {'benign': 69, 'malignant': 31}}
every lesion sits in its image's split: True
raw image ids contain the class label: True -> feature records must use salted pseudonymous ids (bus_research.feature_store.pseudonymize)


## 5. Do genuine BI-RADS reference labels exist?

Search the dataset tree for any annotation file and any field that could hold a BI-RADS assessment,
then run the label gate from `bus_research.birads`.

In [8]:
from bus_research.birads import birads_training_gate

dataset_dir = REPO_ROOT / "dataset"
annotation_like = [p for p in dataset_dir.rglob("*")
                   if p.is_file() and p.suffix.lower() in {".csv", ".json", ".xlsx", ".xls", ".txt", ".xml", ".pdf"}]
birads_mentions = []
for p in annotation_like:
    if "birads" in p.name.lower().replace("-", "").replace(" ", ""):
        birads_mentions.append({"file": str(p.relative_to(REPO_ROOT)), "kind": "filename"})
    elif p.suffix.lower() in {".csv", ".json", ".txt", ".xml"} and p.stat().st_size < 20_000_000:
        text = p.read_text(errors="ignore").lower()
        if "birads" in text.replace("-", "") or "bi-rads" in text:
            birads_mentions.append({"file": str(p.relative_to(REPO_ROOT)), "kind": "content"})
split_fields = sorted({k for rows in splits.values() for row in rows for k in row})
lesion_fields = sorted({k for v in lesions.values() for row in v for k in row})
print("annotation-like files:", len(annotation_like))
print("files mentioning BI-RADS:", birads_mentions)
print("split row fields:", split_fields)
print("lesion row fields:", lesion_fields)

gate = birads_training_gate(labels=None, label_source=None)
print("\nBI-RADS classifier training allowed:", gate.allowed)
for reason in gate.reasons:
    print(" -", reason)

annotation-like files: 654
files mentioning BI-RADS: [{'file': 'dataset/BUSI_Jpeg/BIRADS Reference Card (1).pdf', 'kind': 'filename'}]
split row fields: ['image_id', 'image_path', 'label', 'mask_paths', 'num_lesions']
lesion row fields: ['bbox', 'component_label', 'image_id', 'image_path', 'label', 'source_mask_path']

BI-RADS classifier training allowed: False
 - No BI-RADS reference labels are present in the dataset.


**Finding.** The only BI-RADS artefact is the ACR *Quick Reference* card PDF — a vocabulary
reference, not lesion annotations. There are no BI-RADS assessments, 4A/4B/4C labels, descriptor
annotations, patient ids, dates or clinical metadata. Therefore:

* **Pathway A** (benign vs malignant, from the BUSI class folders) is supported.
* **Pathway B** (BI-RADS category classifier) is **blocked**. Notebook 04 implements the training
  interface and label gate, and stops there. BI-RADS labels are never inferred from benign/malignant
  folders, masks, model probabilities or LLM text.
* The reference card is used only as a retrieval source in notebook 03.

## 6. Save the audit report

In [9]:
from datetime import datetime, timezone

from bus_research.feature_store import atomic_write_json

audit = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "n_images": len(records),
    "images_per_class": dict(label_counts),
    "mask_files_per_image": {str(k): v for k, v in sorted(lesion_counts.items())},
    "images_without_mask": no_mask,
    "orphan_masks": orphan_masks,
    "image_size_px": {"h_min": int(geom.h.min()), "h_max": int(geom.h.max()),
                      "w_min": int(geom.w.min()), "w_max": int(geom.w.max())},
    "physical_pixel_spacing": None,
    "split_sizes": {s: len(rows) for s, rows in splits.items()},
    "split_class_counts": {s: dict(Counter(r["label"] for r in rows)) for s, rows in splits.items()},
    "lesions_per_split": {s: len(v) for s, v in lesions.items()},
    "exact_duplicate_groups": exact_dups,
    "exact_duplicates_cross_split": cross_split_exact,
    "near_duplicate_pairs": near.to_dict("records") if len(near) else [],
    "near_duplicates_cross_split": int(near.cross_split.sum()) if len(near) else 0,
    "near_duplicates_cross_label": int(near.cross_label.sum()) if len(near) else 0,
    "patient_identifiers_available": False,
    "grouping_strategy": "image-level, byte-identical content grouped (dataset/splits.py); no patient ids",
    "raw_ids_contain_label": label_in_id,
    "birads_label_files": birads_mentions,
    "birads_training_gate": gate.to_dict(),
    "pathway_a_supported": True,
    "pathway_b_supported": gate.allowed,
}
path = atomic_write_json(REPO_ROOT / "results/audit/dataset_audit.json", audit)
print("saved", path.relative_to(REPO_ROOT))

saved results/audit/dataset_audit.json


## Limitations

* Without patient ids, image-level grouping is the strongest defensible split. Cross-split
  near-duplicates found above are reported, not removed: removing them would change the split
  that the existing YOLO/U-Net/ResNet50 checkpoints were trained on. Notebook 04 resamples by
  content group in its bootstrap and treats the test estimates as possibly optimistic.
* Near-identical frames filed under *different* class folders exist (count above). They are label
  noise or distinct lesions in the same field of view; without the source annotations this cannot
  be resolved, and they are kept as-is.
* dHash + correlation catches near-identical frames only. Different views of the same lesion
  from the same patient are not detectable from pixels alone.